In [1]:
from reload import reload_package
reload_package(namespace=globals())

Reloaded 0 modules, rebound 0 names.


In [2]:
import altair as alt
import numpy as np
import polars as pl
import torch

from ts_uncertainty.forecasting import ForecastModel
from ts_uncertainty.metrics import mean_absolute_scaled_error, scaled_quantile_loss

#alt.renderers.enable("jupyter", offline=True)
alt.renderers.enable("default")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data

In [3]:
from ts_uncertainty.chaotic import ChaoticGenerator
from ts_uncertainty.injection import Injector
from ts_uncertainty.plotting import (
    _style,
    plot_forecasts,
    plot_sample,
    plot_scores_per_sample,
)
from ts_uncertainty.real import RealSampler
from ts_uncertainty.synthetic import SyntheticGenerator

# data settings
context_length = 256
horizon_length = 128
samples_batch = 100
seed = 42

/home/richard/thesis-ts-uncertainty/.venv/lib/python3.12/site-packages/dysts/base.py:22: UserWarning: Numba not installed. Falling back to no JIT compilation.
  warnings.warn("Numba not installed. Falling back to no JIT compilation.")


## Synthetic Samples

In [4]:
sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    covariate=False,
    periods=(24,),             # period of the sine
    # n_periods=1,
    n_harmonics=3,             # 1 harmonic = pure sine, no extra shape
    # level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    # amplitude_range=(1.0, 1.0),
    seed=seed
)

# generate samples
samples_synthetic = sin_gen.sample_batch(samples_batch)

In [5]:
sample_to_plot = 0
plot_sample(samples_synthetic[sample_to_plot], title=f"Synthetic Sample {sample_to_plot}")

alt.LayerChart(...)

## Chaotic Samples

In [6]:
chaos_gen = ChaoticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    systems=None,                    # None = all ~120 default systems
    dim=0,                           # observed coordinate, None = random per sample
    pts_per_period_range=(20, 30),   # time steps per dominant period (smaller = more irregular)
    level_range=(0.0, 0.0),          # centered at 0
    slope_range=(0.0, 0.0),          # no trend
    amplitude_range=(1.0, 1.0),      # z-normalised signal, std ≈ 1
    seed=seed,
)

# generate samples
samples_chaotic = chaos_gen.sample_batch(samples_batch)

In [7]:
sample_to_plot = 0
plot_sample(samples_chaotic[sample_to_plot], title=f"Chaotic Sample {sample_to_plot}")

alt.LayerChart(...)

## Real Sample
Download the data from the notebook [data_download.ipynb](data_download.ipynb) before.

In [8]:
data_path_test = "../data/test.csv"
data_test = pl.read_csv(data_path_test, try_parse_dates=True)

number_series = len(data_test['series_index'].unique())
print(f"Number of time series in the test set: {number_series}")
data_test.head()

Number of time series in the test set: 361


timestamp,value,series_name,series_index
datetime[μs],f64,str,i64
2025-11-19 10:00:00,0.36,"""home_electricity""",0
2025-11-19 10:15:00,0.124,"""home_electricity""",0
2025-11-19 10:30:00,0.108,"""home_electricity""",0
2025-11-19 10:45:00,0.088,"""home_electricity""",0
2025-11-19 11:00:00,0.116,"""home_electricity""",0


In [9]:
real_gen = RealSampler(
    df=data_test,
    context_length=context_length,
    horizon=horizon_length,
    normalize='none', # dont normalize
    seed=seed,
)

# generate samples
samples_real = real_gen.sample_batch(samples_batch)

In [10]:
sample_to_plot = 0
plot_sample(samples_real[sample_to_plot], title=f"Real Sample {sample_to_plot}")

alt.LayerChart(...)

# Models

In [11]:
models = {
    "AutoTheta": {"name": "autotheta"},
    "Chronos-2": {'name': 'chronos2'},
    "TiRex-2": {'name': 'tirex2'},
    "TimesFM-3": {'name': 'timesfm3'},
    "Timestral (209M)": {'name': 'timestral', 'size': '209M'},
    "ci-proto": {'name': 'ciproto'},
}

# prepare and download all models
for key, m in models.items():
    print(f"Preparing model: {key}")
    model = ForecastModel(**m)

Preparing model: AutoTheta
Preparing model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Preparing model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Preparing model: TimesFM-3
Preparing model: Timestral (209M)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Preparing model: ci-proto


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

# Experiments

In [12]:
def evaluate_models_on_samples(models, samples, clean_samples=None):
    """clean_samples: the same samples before injection (same order); their context gives the
    MASE / SQL scale, so an injection in the context doesn't change the denominator."""
    results = []
    scores_mase = []
    scores_sql = []
    scores_mase_per_sample = []
    scores_sql_per_sample = []   

    for key, m in models.items():
        print(f"Evaluating model: {key}")
        model = ForecastModel(**m)
        fc = model.forecast(samples)

        fc_m = fc.metric_inputs()
        if clean_samples is None:
            y_scale = fc_m["y_train"]  # (T, B)
        else:
            y_scale = np.stack([s.context for s in clean_samples], axis=1)  # (T, B)

        score_mase = mean_absolute_scaled_error(fc_m["y_true"], fc_m["y_pred"], y_scale, fc_m["season_length"])
        score_sql = scaled_quantile_loss(fc_m["y_true"], fc_m["y_quantiles"], y_scale, fc_m["quantiles"], fc_m["season_length"])

        results.append(fc)
        scores_mase.append(score_mase.mean())
        scores_sql.append(score_sql.mean())
        scores_mase_per_sample.append(score_mase)
        scores_sql_per_sample.append(score_sql)

    scores = pl.DataFrame({
        "model": list(models.keys()),
        "mase": [float(x.squeeze()) for x in scores_mase],
        "sql": [float(x.squeeze()) for x in scores_sql],
    })
    return results, scores, scores_mase_per_sample, scores_sql_per_sample


def summarize_scores(scores: dict) -> pl.DataFrame:
    """Long score table, one row per (dataset, model).

    scores: either one run {dataset: DataFrame}, giving columns mase, sql, ...;
    or several runs {run: {dataset: DataFrame}}, giving columns mase_<run>, sql_<run>, ...
    """
    def to_long(fc_scores: dict[str, pl.DataFrame]) -> pl.DataFrame:
        long = pl.concat([df.with_columns(pl.lit(name).alias("dataset")) for name, df in fc_scores.items()])
        return long.select("dataset", "model", pl.exclude("dataset", "model"))

    if all(isinstance(v, pl.DataFrame) for v in scores.values()):  # single run
        return to_long(scores)

    tables = []
    for run, fc_scores in scores.items():
        long = to_long(fc_scores)
        tables.append(long.select("dataset", "model", *[pl.col(m).alias(f"{m}_{run}") for m in long.columns[2:]]))

    out = tables[0]
    for t in tables[1:]:
        out = out.join(t, on=["dataset", "model"], how="full", coalesce=True, maintain_order="left")

    # group columns by metric: mase_clean, mase_noise_h, ..., sql_clean, sql_noise_h, ...
    runs = list(scores)
    metric_cols = sorted(out.columns[2:], key=lambda c: (c.split("_", 1)[0], runs.index(c.split("_", 1)[1])))
    return out.select("dataset", "model", *metric_cols)


def delta_scores(summary: pl.DataFrame, reference: str = "clean") -> pl.DataFrame:
    """Mean degradation vs. the reference run: <metric>_<run> - <metric>_<reference>, one row per (dataset, model)."""
    metrics = sorted({c.rsplit("_", 1)[0] for c in summary.columns if c.endswith(f"_{reference}")})
    deltas = [
        (pl.col(c) - pl.col(f"{m}_{reference}")).alias(f"d_{c}")
        for m in metrics
        for c in summary.columns
        if c.startswith(f"{m}_") and c != f"{m}_{reference}"
    ]
    return summary.select("dataset", "model", *deltas)


In [13]:
batch_size = 1  # define the batch size for evaluation
data_samples = {
    "synthetic": samples_synthetic[:batch_size],
    "chaotic": samples_chaotic[:batch_size],
    "real": samples_real[:batch_size],
}

## No Injection

In [14]:
fc_results = {}
fc_scores = {}
for type_s, s in data_samples.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores, scores_mase_per_sample, scores_sql_per_sample = evaluate_models_on_samples(models, s)
    # save results and scores for a dataset
    fc_results[type_s] = results
    fc_scores[type_s] = scores

Forecasting synthetic samples..., number of samples: 1
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

ninja: no work to do.


... (25 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: ci-proto


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=3):
    print(summarize_scores(fc_scores))

shape: (18, 4)
┌───────────┬──────────────────┬───────┬───────┐
│ dataset   ┆ model            ┆ mase  ┆ sql   │
│ ---       ┆ ---              ┆ ---   ┆ ---   │
│ str       ┆ str              ┆ f64   ┆ f64   │
╞═══════════╪══════════════════╪═══════╪═══════╡
│ synthetic ┆ AutoTheta        ┆ 0.000 ┆ 4.829 │
│ synthetic ┆ Chronos-2        ┆ 0.078 ┆ 0.059 │
│ synthetic ┆ TiRex-2          ┆ 0.123 ┆ 0.143 │
│ synthetic ┆ TimesFM-3        ┆ 0.061 ┆ 0.048 │
│ synthetic ┆ Timestral (209M) ┆ 0.030 ┆ 0.024 │
│ synthetic ┆ ci-proto         ┆ 0.043 ┆ 0.038 │
│ chaotic   ┆ AutoTheta        ┆ 1.670 ┆ 3.287 │
│ chaotic   ┆ Chronos-2        ┆ 3.409 ┆ 2.260 │
│ chaotic   ┆ TiRex-2          ┆ 3.079 ┆ 2.097 │
│ chaotic   ┆ TimesFM-3        ┆ 3.455 ┆ 2.403 │
│ chaotic   ┆ Timestral (209M) ┆ 3.593 ┆ 2.748 │
│ chaotic   ┆ ci-proto         ┆ 4.239 ┆ 3.083 │
│ real      ┆ AutoTheta        ┆ 0.963 ┆ 0.766 │
│ real      ┆ Chronos-2        ┆ 0.642 ┆ 0.490 │
│ real      ┆ TiRex-2          ┆ 0.875 ┆ 0.642 │
│ rea

In [16]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
sample_to_plot = 0

forecasts = dict(zip(models.keys(), fc_results[dataset_to_plot]))
plot_forecasts(
    data_samples[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_57690/250352761.py:5: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


alt.ConcatChart(...)

## Shift

### Without covariates

In [17]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate=False,
    spike_covariate=False,
    shift_region="any",
    shift_position_range=(0.0, 1.0),
    shift_level_range = (1.0, 1.0),
    shift_slope_range = (0.02, 0.02),
    shift_scale_range = (1.0, 1.0),    
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
shift_points = np.round(np.linspace(0.0, 1.0, context_length + horizon_length), 4)
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real",
                                        shift_point=sp,
                                        shift_sign=1,
                                        shift_type="trend") for x in s for sp in shift_points]

fc_results_shift_w = {}
fc_scores_shift_w = {}
fc_scores_mase_per_sample_shift_w = {}
fc_scores_sql_per_sample_shift_w = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores, scores_mase_per_sample, scores_sql_per_sample = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s]*len(shift_points))
    # save results and scores for a dataset
    fc_results_shift_w[type_s] = results
    fc_scores_shift_w[type_s] = scores
    fc_scores_mase_per_sample_shift_w[type_s] = scores_mase_per_sample
    fc_scores_sql_per_sample_shift_w[type_s] = scores_sql_per_sample

Injecting anomalies into synthetic samples..., number of samples: 1
Injecting anomalies into chaotic samples..., number of samples: 1
Injecting anomalies into real samples..., number of samples: 1
Forecasting synthetic samples..., number of samples: 384
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: TimesFM-3
Evaluating model: Timestral (209M)


... (23 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: ci-proto


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [18]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
plot_score_sample_sql = plot_scores_per_sample(
    fc_scores_sql_per_sample_shift_w[dataset_to_plot],
    model_names=list(models.keys()),
    x_title="time step of the shift point",
    y_title="SQL",
    title=f"SQL per shift point ({dataset_to_plot})",
    forecast_start=context_length,
    log_y=True,                # if AutoTheta's errors squash the others
)
plot_score_sample_mase = plot_scores_per_sample(
    fc_scores_mase_per_sample_shift_w[dataset_to_plot],
    model_names=list(models.keys()),
    x_title="time step of the shift point",
    y_title="MASE",
    title=f"MASE per shift point ({dataset_to_plot})",
    forecast_start=context_length,
    # log_y=True,                # if AutoTheta's errors squash the others
)

for c in (plot_score_sample_sql, plot_score_sample_mase):
    c.config = alt.Undefined  # styling goes on the combined chart
combined = (plot_score_sample_sql & plot_score_sample_mase).properties(
    title=alt.Title(
        f"Errors per shift point (without covariate)",
        anchor="start",  # left-aligned like the panel titles; "middle" to centre
    )
)
_style(combined)

/tmp/ipykernel_57690/2213991161.py:23: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  combined = (plot_score_sample_sql & plot_score_sample_mase).properties(


alt.VConcatChart(...)

In [19]:
sample_to_plot = 220

forecasts = dict(zip(models.keys(), fc_results_shift_w[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_57690/650626177.py:4: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


alt.ConcatChart(...)

### With covariates (binary)

In [20]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate=True,
    spike_covariate=False,
    shift_region="any",
    shift_position_range=(0.0, 1.0),
    shift_level_range = (1.0, 1.0),
    shift_slope_range = (0.02, 0.02),
    shift_scale_range = (1.0, 1.0),    
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
shift_points = np.round(np.linspace(0.0, 1.0, context_length + horizon_length), 4)
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real",
                                        shift_point=sp,
                                        shift_sign=1,
                                        shift_type="trend") for x in s for sp in shift_points]

fc_results_shift_w = {}
fc_scores_shift_w = {}
fc_scores_mase_per_sample_shift_w = {}
fc_scores_sql_per_sample_shift_w = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores, scores_mase_per_sample, scores_sql_per_sample = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s]*len(shift_points))
    # save results and scores for a dataset
    fc_results_shift_w[type_s] = results
    fc_scores_shift_w[type_s] = scores
    fc_scores_mase_per_sample_shift_w[type_s] = scores_mase_per_sample
    fc_scores_sql_per_sample_shift_w[type_s] = scores_sql_per_sample

Injecting anomalies into synthetic samples..., number of samples: 1
Injecting anomalies into chaotic samples..., number of samples: 1
Injecting anomalies into real samples..., number of samples: 1
Forecasting synthetic samples..., number of samples: 384
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: TimesFM-3
Evaluating model: Timestral (209M)


... (23 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: ci-proto


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [21]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
plot_score_sample_sql = plot_scores_per_sample(
    fc_scores_sql_per_sample_shift_w[dataset_to_plot],
    model_names=list(models.keys()),
    x_title="time step of the shift point",
    y_title="SQL",
    title=f"SQL per shift point ({dataset_to_plot})",
    forecast_start=context_length,
    log_y=True,                # if AutoTheta's errors squash the others
)
plot_score_sample_mase = plot_scores_per_sample(
    fc_scores_mase_per_sample_shift_w[dataset_to_plot],
    model_names=list(models.keys()),
    x_title="time step of the shift point",
    y_title="MASE",
    title=f"MASE per shift point ({dataset_to_plot})",
    forecast_start=context_length,
    # log_y=True,                # if AutoTheta's errors squash the others
)

for c in (plot_score_sample_sql, plot_score_sample_mase):
    c.config = alt.Undefined  # styling goes on the combined chart
combined = (plot_score_sample_sql & plot_score_sample_mase).properties(
    title=alt.Title(
        "Errors per shift point (with binary covariate)",
        anchor="start",  # left-aligned like the panel titles; "middle" to centre
    )
)
_style(combined)

/tmp/ipykernel_57690/2973793093.py:23: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  combined = (plot_score_sample_sql & plot_score_sample_mase).properties(


alt.VConcatChart(...)

In [22]:
sample_to_plot = 155

forecasts = dict(zip(models.keys(), fc_results_shift_w[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_57690/543796564.py:4: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


alt.ConcatChart(...)

### With covariates (ramp)

In [23]:
inj = Injector(
    noise=False,
    spikes=False,
    shift=True,
    shift_covariate='ramp',
    spike_covariate=False,
    shift_region="any",
    shift_position_range=(0.0, 1.0),
    shift_level_range = (1.0, 1.0),
    shift_slope_range = (0.02, 0.02),
    shift_scale_range = (1.0, 1.0),    
    seed=seed,
)

# inject anomalies into the samples
data_samples_injected = {}
shift_points = np.round(np.linspace(0.0, 1.0, context_length + horizon_length), 4)
for type_s, s in data_samples.items():
    print(f"Injecting anomalies into {type_s} samples..., number of samples: {len(s)}")
    data_samples_injected[type_s] = [inj(x, relative=type_s == "real",
                                        shift_point=sp,
                                        shift_sign=1,
                                        shift_type="trend") for x in s for sp in shift_points]

fc_results_shift_wr = {}
fc_scores_shift_wr = {}
fc_scores_mase_per_sample_shift_wr = {}
fc_scores_sql_per_sample_shift_wr = {}
for type_s, s in data_samples_injected.items():
    print(f"Forecasting {type_s} samples..., number of samples: {len(s)}")
    results, scores, scores_mase_per_sample, scores_sql_per_sample = evaluate_models_on_samples(models, s, clean_samples=data_samples[type_s]*len(shift_points))
    # save results and scores for a dataset
    fc_results_shift_wr[type_s] = results
    fc_scores_shift_wr[type_s] = scores
    fc_scores_mase_per_sample_shift_wr[type_s] = scores_mase_per_sample
    fc_scores_sql_per_sample_shift_wr[type_s] = scores_sql_per_sample

Injecting anomalies into synthetic samples..., number of samples: 1
Injecting anomalies into chaotic samples..., number of samples: 1
Injecting anomalies into real samples..., number of samples: 1
Forecasting synthetic samples..., number of samples: 384
Evaluating model: AutoTheta
Evaluating model: Chronos-2


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Evaluating model: TiRex-2


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: TimesFM-3
Evaluating model: Timestral (209M)


... (23 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating model: ci-proto


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [24]:
dataset_to_plot = "synthetic"  # choose from "synthetic", "chaotic", "real"
plot_score_sample_sql = plot_scores_per_sample(
    fc_scores_sql_per_sample_shift_wr[dataset_to_plot],
    model_names=list(models.keys()),
    x_title="time step of the shift point",
    y_title="SQL",
    title=f"SQL per shift point ({dataset_to_plot})",
    forecast_start=context_length,
    log_y=True,                # if AutoTheta's errors squash the others
)
plot_score_sample_mase = plot_scores_per_sample(
    fc_scores_mase_per_sample_shift_wr[dataset_to_plot],
    model_names=list(models.keys()),
    x_title="time step of the shift point",
    y_title="MASE",
    title=f"MASE per shift point ({dataset_to_plot})",
    forecast_start=context_length,
    # log_y=True,                # if AutoTheta's errors squash the others
)

for c in (plot_score_sample_sql, plot_score_sample_mase):
    c.config = alt.Undefined  # styling goes on the combined chart
combined = (plot_score_sample_sql & plot_score_sample_mase).properties(
    title=alt.Title(
        "Errors per shift point (with effect covariate)",
        anchor="start",  # left-aligned like the panel titles; "middle" to centre
    )
)
_style(combined)

/tmp/ipykernel_57690/3010294398.py:23: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  combined = (plot_score_sample_sql & plot_score_sample_mase).properties(


alt.VConcatChart(...)

In [25]:
sample_to_plot = 273

forecasts = dict(zip(models.keys(), fc_results_shift_wr[dataset_to_plot]))
plot_forecasts(
    data_samples_injected[dataset_to_plot][sample_to_plot],
    forecasts,
    index=sample_to_plot,
    exclude_from_y=[list(models)[0]],  # AutoTheta: its wide bands would squash the plot
)

/tmp/ipykernel_57690/96635270.py:4: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  plot_forecasts(


alt.ConcatChart(...)